<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

# Set variables

In [1]:
GROUND_TRUTH_ROOT = "AudioMNIST-master/data"
SENSOR_ROOT = "AudioMNIST-master/gen"
archivo_csv= "archivo_csv.csv"
seed = 42

In [2]:
# Parameters
archivo_csv = "AudioMNIST-master/gen/csv/01/1_01_13.csv"


## Import Needed Libraries

In [3]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import *
import os
import torchinfo
import torchvision.transforms

## Import our own backend

In [4]:
import core.nn.AudioMNIST
from core.signal.preprocess import *
from core.nn.utils import normalize, denormalize, mapToBounds, apply_wiener
import whisper

## Set seed and logging level

In [5]:
import logging
# fix seed for reproducibility
np.random.seed(seed)
torch.manual_seed(seed)
logging.basicConfig(level=logging.INFO)

## Initialize the dataloaders

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
Fs = 16000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds = core.nn.AudioMNIST.audioMnistDataset(basedir="AudioMNIST-master", device = device, Fs=Fs)
train_loader, test_loader, dev_loader = core.nn.AudioMNIST.getAudioLoaders(ds, batch_size=32, device = device, ret_labels=False)

ValueError: num_samples should be a positive integer value, but got num_samples=0

## Load in the Noise data and build the Wiener Filter

In [ ]:
from math import ceil
_, Xnoise, Ynoise = processFromFile("AudioMNIST-master/gen/noise/noise_2023-11-03_21-00-40.csv", method='sinc')
# Pad to closest 16000
Fs = 16000
noise = F.pad((Xnoise+Ynoise)/2, (0, ceil((Xnoise.shape[0] / (Fs)))*( Fs) - Xnoise.shape[0]), "constant", 0)
noise = noise.reshape(-1, Fs)

In [ ]:
# compute FFT in dev set
Data_PSD = []
Noisy_PSD = []
for Xorig, Yorig in tqdm(dev_loader):
    # Get FFTs and aggregate
    Yorig = torch.fft.rfft(Yorig.squeeze().reshape(-1, Fs).to(device), dim = -1)
    Xorig = torch.fft.rfft(Xorig.squeeze().reshape(-1, 2, Fs).to(device), dim = -1)
    Data_PSD.append(Yorig)
    Noisy_PSD.append(Xorig)
# Compute average PSD
avgData_PSD = torch.cat(Data_PSD, dim=0).abs().mean(dim=0).cpu()
avgNoisy_PSD = torch.cat(Noisy_PSD, dim=0).abs().mean(dim=0).cpu().mean(dim=0)
# Compute FFT
noiseFFT = torch.fft.rfft(noise, dim=1)
noiseFFT = torch.abs(noiseFFT)
noiseFFT = noiseFFT.mean(dim=0)
H = avgData_PSD / (avgData_PSD + noiseFFT)
plt.plot(H.abs())

## Prepare the transforms to and from the spectrogram domain

In [ ]:
import einops
tforms = core.nn.utils.buildTransforms(device = device, 
                                        Fs = Fs, 
                                        n_fft = 800, 
                                        win_length = 800,
                                        hop_length = 160,
                                        n_mels = 80,
                                        f_max_mouse = 8000,
                                        f_max_full = 8000,
                                        )
specFn, inverseSpecFn, melFilterMouse, melFilter, inverseMelFn = tforms

toMelDB = lambda batch: core.nn.utils.toMelDB(batch, specFn=specFn, mousemelfilters=melFilterMouse, fullmelfilter=melFilter)

def computeUnboundedSpec(X):
    ms = True if X.shape[1] == 2 else False
    if ms:
        X -= einops.reduce(X, "b c t -> b c ()", "mean")  # Remove mean
    else:
        X -= einops.reduce(X, "b t -> b ()", "mean")  # Remove mean
    X /= (torch.std(X) + 1e-8)                         # Normalize                                        
    X = specFn(X)                                 # Convert to mel
    X = X.abs().pow(2)                           # Power
    
    if ms:
        X = torch.einsum("bkct,fc->bkft", X, melFilterMouse)
        X = einops.rearrange(X, "b c m t -> b (m c) t", c=2)
    else:
        X = torch.einsum("bct,fc->bft", X, melFilter) # Apply mel filter
        
    X = (torch.maximum(torch.clamp(X, min=1e-10).log10(), torch.clamp(X, min=1e-10).log10().max() - 8.0) + 4.0)/ 4.0    # Convert to dB
    return X

# # The following code is used to compute the min and max values of the spectrograms

mnmn = torch.inf
mxmx = -torch.inf
for Mwav, Wwav in dev_loader:
    MwavFilt = apply_wiener(Mwav, H.to(device))
    Mfilt = computeUnboundedSpec(MwavFilt)
    M = computeUnboundedSpec(Mwav)
    W = computeUnboundedSpec(Wwav)
    mnmn = min(mnmn, M.min(), W.min(), Mfilt.min())
    mxmx = max(mxmx, M.max(), W.max(), Mfilt.max())

import einops


def computeSpec(X):
    X = computeUnboundedSpec(X)
    X = (X - mnmn) / (mxmx - mnmn)               # Map to bounds
    X = torch.clamp(X, 0, 1)                    # Clamp to [0, 1]
    X = 2 * X - 1                               # Map to [-1, 1]
    return X

def fromSpec(X):
    if X.shape[1] == 160:
        X = einops.rearrange(X, "b (m c) t -> b c m t", c=2)
        X = torch.mean(X, dim=1)
    X = (X + 1) / 2
    X = X * (mxmx - mnmn) + mnmn
    X = 10 ** (4 * X - 4)
    X = inverseMelFn(X)
    X = inverseSpecFn(X)
    return X

### Plot the spectrum of the sensor data, ground truth, and post-Wiener Filter sensor data

In [ ]:
postFiltering = H.abs() * avgNoisy_PSD.abs()
# fft bin to frequency
X = torch.fft.rfftfreq(16000, 1/16000)

# build a dataframe with 4 columns: frequency, avgNoisy, avgData_PSD, postFiltering
df = pd.DataFrame()
df['frequency'] = X
df['avgNoisy'] = avgNoisy_PSD
df['avgData'] = avgData_PSD
df['postFiltering'] = postFiltering
# save the dataframe
df.to_csv('dataDist.csv', index=False)
plt.plot(postFiltering)
plt.plot(avgData_PSD)

### Load in the Whisper model (encoder only)

In [ ]:
perceptualEmbedding = whisper.load_model("tiny").encoder.to(device)
perceptualEmbedding.eval()
for p in perceptualEmbedding.parameters():
    p.requires_grad = False

### Define the filtering NN

In [ ]:
from core.nn.whisperPrimitives import *
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embed_d = 384
n_head = 6
n_layer = 4
k_size = 15
padding = k_size // 2
n_mels = 80
model = nn.Sequential(
    nn.Conv1d(n_mels * 2, embed_d, k_size, padding=padding),
    nn.GELU(),
    TransformerChain(n_ctx=101, n_state=embed_d, n_head = n_head, n_layer=n_layer),
    nn.GELU(),
    nn.Conv1d(embed_d, n_mels * 4, k_size, padding=padding),
    nn.GELU(),
    nn.Conv1d(n_mels * 4, n_mels, 1, padding=0),
).to(device)

## Define Loss Functions, Optimizer, and Scheduler

In [ ]:
def frobenius_norm(X):
    return torch.sqrt(torch.sum(X ** 2))

def spectral_convergence(X, Y):
    return frobenius_norm(X - Y) / frobenius_norm(Y)

lr = 5e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.707)

l1_loss = nn.L1Loss()
l2_loss = nn.MSELoss()
huber_loss = nn.HuberLoss()
perceptual_loss = lambda x, y: l2_loss(perceptualEmbedding(x), perceptualEmbedding(y))
pad_fn = lambda x, y: (F.pad(x, (0, 3000 - x.shape[-1])), F.pad(y, (0, 3000 - y.shape[-1])))

# Uncomment the losses you want to use, Huber loss is the most stable, perceptual loss is the most effective but takes the longest to train, we use a weighted sum of the losses for now
loss_fn = lambda x, y: (
    # l2_loss(x, y) +
    # l1_loss(x, y) +
    # perceptual_loss(*pad_fn(x, y)) +  # For AudioMNIST, we found that training without this loss yields better results
    huber_loss(x, y)
)

### Training Helpers

In [ ]:
def update_model(model, batch):
    model.train()
    optimizer.zero_grad()
    X = batch[0].to(device)
    Y = batch[1].to(device)
    
    # Apply wiener filter, normalize, convert to mel, map to bounds
    X = apply_wiener(X, H.to(device))
    X = computeSpec(X)
    Y = computeSpec(Y)
    
    # Train
    recon_batch = model(X)
    loss = loss_fn(recon_batch, Y)
    loss.backward()
    optimizer.step()
    return loss.item()

def train(model, loader, epochs = 10):
    model.train()
    with tqdm(total = epochs * len(loader)) as pbar:
        for epoch in range(epochs):
            for batch_idx, batch in enumerate(loader):
                loss = update_model(model, batch)
                pbar.set_description(f"Epoch {epoch}/{epochs}\tBatch {batch_idx}/{len(loader)}\tLoss {loss}")
                pbar.update(1)
                # wandb.log({"loss": loss}, step = epoch * len(loader) + batch_idx)
            # save model
            scheduler.step()
            torch.save(model.state_dict(), f"temp/model_SUM_E{epoch}_L{loss:.2f}.pt")

### Train

In [ ]:
model.load_state_dict(torch.load("models/model_AUDIOMNIST.model", map_location=torch.device('cpu')))

In [ ]:
#model.load_state_dict(torch.load("models/model_AUDIOMNIST.model"))
#train(model, train_loader, epochs = 150)

In [ ]:
torch.save(model.state_dict(), "models/model_AUDIOMNIST.model")

### Show one sample

In [ ]:
# run one test batch
model.eval()

with torch.no_grad():
    Xraw, Y = next(iter(dev_loader))
    Xfilt = apply_wiener(Xraw, H.to(device))
    Xraw = computeSpec(Xraw)
    Xfilt = computeSpec(Xfilt)
    Y = computeSpec(Y)

    recon_batch = model(Xfilt)

fig, ax = plt.subplots(1, 4, figsize=(7, 1.5))
i = 0
ax[0].imshow(Y[i, :, :301].cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
ax[0].set_xticks([]); ax[0].set_yticks([])
ax[1].imshow(Xraw[i, :, :301].cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
ax[1].set_xticks([]); ax[1].set_yticks([])
ax[2].imshow(Xfilt[i, :, :301].cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
ax[2].set_xticks([]); ax[2].set_yticks([])
ax[3].imshow(recon_batch[i, :, :301].cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
ax[3].set_xticks([]); ax[3].set_yticks([])
fig.tight_layout()
plt.show()

### Use first batch from test loader as human study

In [ ]:
ModelOutputs = []
WienerOutputs = []
GrountTruth = []
i = 0
model.load_state_dict(torch.load("models/model_AUDIOMNIST.model"))
with torch.no_grad():
    for X, Y in tqdm(test_loader):
        X, Y = next(iter(test_loader))
        X = apply_wiener(X, H.to(device))
        WienerOutputs.append(X.mean(dim=1))
        Y = computeSpec(Y)
        X = computeSpec(X)
        X = model(X.to(device))
        X = fromSpec(X)
        Y = fromSpec(Y)
        ModelOutputs.append(X)
        GrountTruth.append(Y)
        i += 1
        if i == 1:
            break
PerceptOutputs = []
#model.load_state_dict(torch.load("models/model_AUDIOMNIST_PERCEPT.model"))
model.load_state_dict(torch.load("models/model_AUDIOMNIST_PERCEPT.model", map_location=torch.device('cpu')))
with torch.no_grad():
    X, Y = next(iter(test_loader))
    X = apply_wiener(X, H.to(device))
    X = computeSpec(X)
    X = model(X.to(device))
    X = fromSpec(X)
    PerceptOutputs.append(X)
model.load_state_dict(torch.load("models/model_AUDIOMNIST.model"))

In [ ]:
ModelOutputs = torch.cat(ModelOutputs, dim=0) # 
WienerOutputs = torch.cat(WienerOutputs, dim=0)
PerceptOutputs = torch.cat(PerceptOutputs, dim=0)
GrountTruth = torch.cat(GrountTruth, dim=0)

In [ ]:
from IPython.display import Audio
XY = torch.cat([ModelOutputs, torch.zeros(32, 8000, device=device), GrountTruth], dim=1)
XY = F.pad(XY, (0, 10 * Fs - XY.shape[1]), "constant", 0)
torchaudio.save("out/ModelTest.wav", XY.view(1, -1).cpu(), Fs)
Audio(XY.view(-1).cpu(), rate=Fs)

In [ ]:
OY = torch.cat([PerceptOutputs, torch.zeros(32, 8000, device=device), GrountTruth], dim=1)
OY = F.pad(OY, (0, 10 * Fs - OY.shape[1]), "constant", 0)
torchaudio.save("out/ModelPERCEPTTest.wav", OY.view(1, -1).cpu(), Fs)
Audio(OY.view(-1).cpu(), rate=Fs)

In [ ]:
WY = torch.cat([WienerOutputs * 100, torch.zeros(32, 8000, device=device), GrountTruth], dim=1)
WY = F.pad(WY, (0, 10 * Fs - WY.shape[1]), "constant", 0)
torchaudio.save("out/WienerTest.wav", WY.view(1, -1).cpu(), Fs)
Audio(WY.view(-1).cpu(), rate=Fs)

# Prepare the video presentation

In [ ]:
_, Xvid, Yvid = processFromFile(archivo_csv, method='sinc')
Xvid = torch.stack([Xvid, Yvid], dim=0)
X = F.pad(Xvid, (0, ceil(Xvid.shape[-1] / Fs)*( Fs) - Xvid.shape[-1]), "constant", 0)
X = X.reshape(2, -1, Fs)
X = X.permute(1, 0, 2)
# Take all the odd samples
X = X.to(device)
Xp = apply_wiener(X, H.to(device))
torchaudio.save("resultados/RawOutput.wav", X.mean(dim = 1).view(-1).cpu().unsqueeze(0), sample_rate = 16000)
Audio(X.mean(dim = 1).view(-1).cpu(), rate= Fs)

In [ ]:
Xspec = computeSpec(X)
plt.figure(figsize=(20, 8))
Xspec = einops.rearrange(Xspec, "b c t -> c (b t)")
plt.imshow(Xspec.cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
# remove axis
plt.axis(False)
plt.tight_layout()
plt.savefig("resultados/RawOutput.png", dpi = 400)

In [ ]:
torchaudio.save("resultados/WienerOutput.wav", Xp.mean(dim = 1).view(-1).cpu().unsqueeze(0), sample_rate = 16000)
Audio(Xp.mean(dim = 1).view(-1).cpu(), rate= Fs)

In [ ]:
Xspec = computeSpec(Xp)
plt.figure(figsize=(20, 8))
Xspec = einops.rearrange(Xspec, "b c t -> c (b t)")
plt.imshow(Xspec.cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
# remove axis
plt.axis(False)
plt.tight_layout()
plt.savefig("resultados/WienerOutput.png", dpi = 400)

In [ ]:
L =[ # instead of trying to align to the video's timing, we use the previously generated samples
    archivo_csv,
]
L = [processFromFile(e, method='sinc') for e in L]
L = [torch.stack([e[1], e[2]], dim=0) for e in L]
L = [F.pad(e, (0, 16000 - e.shape[-1])) for e in L]
L = torch.stack(L, dim = 0)
X = L.to(device)
Xp = apply_wiener(X, H.to(device))
Xp = computeSpec(Xp)
with torch.no_grad():
    Y = model(Xp)
Y = fromSpec(Y)
Yl = []
for e in Y:
    Yl.append(e.cpu())
    Yl.append(torch.zeros(16000))
Y = torch.cat(Yl, dim = 0)

In [ ]:
torchaudio.save("resultados/ModelOutput.wav", Y.unsqueeze(0), sample_rate = 16000)
Audio(Y, rate = 16000)

In [ ]:
Xspec = computeSpec(Y.unsqueeze(0).to(device))
Xspec = einops.rearrange(Xspec, "b c t -> c (b t)")
plt.figure(figsize=(20, 8))
plt.imshow(Xspec.cpu(), aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="magma")
# remove axis
plt.axis(False)
plt.tight_layout()
plt.savefig("resultados/ModelOutput.png", dpi=400)